# CTG Signal Explorer
Load a single patient and compare raw vs processed signals.

**`plot_patient_signals`** — HR1 left panel / TOCO right panel, raw vs processed overlaid per panel  
**`plot_patient_signals_post`** — FHR + TOCO on the same axis, raw panel left / processed panel right, birth time marked

In [ ]:
import sys
sys.path.insert(0, ".")

import pandas as pd

from config import load_config
from loader import load_patient_by_id
from preprocess import preprocess_patient
from visualize import plot_patient_signals, plot_patient_signals_post

In [ ]:
config = load_config("config.yaml")

### Pick a patient

In [ ]:
stats = pd.read_csv("output/per_patient_stats.csv")
stats[["patient_id", "duration_post_trim_min", "hr1_outliers", "toco_outliers"]].head(10)

In [ ]:
# Change this to any patient_id from the table above
PATIENT_ID = "0x01000000004756075E1F49B0952786CDDCFBA310E2C9C0C5F818727E7AA48CB346CE725A_2010"

### Load & preprocess

In [ ]:
# Strip trailing session suffix (_0, _1, …) if present
base_id = PATIENT_ID.rsplit("_", 1)[0] if (PATIENT_ID[-2] == "_" and PATIENT_ID[-1].isdigit()) else PATIENT_ID

raw_df = load_patient_by_id(base_id, config)
if raw_df is None:
    raise ValueError(f"No data found for patient: {base_id}")

print(f"Loaded {len(raw_df):,} rows  |  "
      f"{raw_df['Monitor_Date'].min()}  →  {raw_df['Monitor_Date'].max()}")

In [ ]:
processed_df, stats_row = preprocess_patient(raw_df, config, patient_id=PATIENT_ID)

print(
    f"Duration:  {stats_row.duration_pre_trim_minutes:.1f} min raw  →  "
    f"{stats_row.duration_post_trim_minutes:.1f} min trimmed  →  "
    f"{stats_row.duration_post_resample_minutes:.1f} min resampled\n"
    f"Outliers:  HR1={stats_row.n_hr1_outliers}  TOCO={stats_row.n_toco_outliers}\n"
    f"Interpolated: {stats_row.n_interpolated:,} samples"
)

### Plot 1 — HR1 vs TOCO side-by-side (raw vs processed per panel)
Rangeslider beneath the left panel zooms both panels simultaneously.

In [ ]:
fig1 = plot_patient_signals(PATIENT_ID, raw_df, processed_df)
fig1.show()

### Plot 2 — FHR + TOCO on same axis, raw left / processed right
Set `BIRTH_TIME` to the actual delivery timestamp for this patient.

In [ ]:
# Replace with the real birth timestamp for this patient
BIRTH_TIME = pd.Timestamp("2010-07-08 06:30:00")

fig2 = plot_patient_signals_post(PATIENT_ID, BIRTH_TIME, raw_df, processed_df)
fig2.show()